# Worked solutions · Sections 6 and 7 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of each section,
as F1–F5 and G1–G6.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
Theorem 1.3.43 (`thm.dassiosZhao`), eq. (1.3.41, `eq.dzDraws`).
Type 0 is the market buys and type 1 the market sells.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebooks `06-simulating-hawkes` and `07-goodness-of-fit` run them and verify every key.

In [ ]:
from math import exp, log, pi, sqrt

# Section 6 · Simulating a Hawkes process

**F1. Branching matrix and column sums.**
For the exponential kernel $\Gamma = A/\beta$, eq. (1.3.12, `eq.GammaIsAOverBeta`).
The column sum $\sum_e A_{e,e'}$ is the jump of the total intensity $\lambda_{\mathfrak g}$
caused by one event of type $e'$;
the column sum $\sum_e \Gamma_{e,e'}$ is the expected number of its direct offspring.
They differ by the factor $\beta$, Remark 1.3.10 (`remark.countsAgainstRates`).

**F2. Branching ratio.**
$\rho$ is the spectral radius of $\Gamma$ (Section 1.3.2 (`sec.hawkesProcesses`), after eq. (1.3.9, `eq.branchingMatrix`)).
Lemma 1.3.17 (`lemma.perronFrobenius`), part iii, applied to $\Gamma^\top$
bounds $\rho$ between the smallest and the largest column sum of $\Gamma$
(the paragraph after the lemma):
constant column sums $c$ give $\rho = c$.

**F3. Total rate.**
Under $\rho < 1$ and $\mu > 0$, Theorem 1.3.19 (`thm.stationarity`) gives
$$\lambda^* = (I - \Gamma)^{-1}\mu,$$
eq. (1.3.18, `eq.stationaryIntensity`).
The total rate is $\bar\lambda = \mathbf 1^\top\lambda^*$
and the total immigration rate $\bar\mu = \mathbf 1^\top\mu$
(the definitions after Remark 1.3.20 (`remark.strictBaseline`)).
When $\mathbf 1^\top\Gamma = \rho\,\mathbf 1^\top$,
Corollary 1.3.25 (`corol.constantClusterSize`) gives $\mathbf 1^\top(I-\Gamma)^{-1} = \mathbf 1^\top/(1-\rho)$, hence
$$\bar\lambda = \mathbf 1^\top(I-\Gamma)^{-1}\mu = \frac{\bar\mu}{1-\rho}.$$
This is eq. (1.3.23, `eq.meanClusterSize`), $\bar\lambda = \bar\mu\,\bar C$, with $\bar C = 1/(1-\rho)$.
A simulation started from $Z = 0$ counts $\bar\lambda$ events per second
over a horizon long against the relaxation time $1/(\beta(1-\rho))$ of Definition 1.3.36 (`def.relaxationTime`).

**F4. Intensity and state.**
$\lambda = \mu + AZ$, eq. (1.3.13, `eq.intensityFromState`),
so $\lambda_{\mathfrak g} = \mathbf 1^\top\lambda = \bar\mu + (\mathbf 1^\top A)\,Z$.
The state $Z$ of eq. (1.3.11, `eq.decayedCounts`) sums over the events strictly before $t$,
so it is left-continuous:
at an arrival $T_n$ of type $E_n$, $Z(T_n)$ is the value before the jump, the one the intensity uses,
and $Z(T_n+) = Z(T_n) + \mathbf e_{E_n}$ the value carried forward, Remark 1.3.41 (`remark.prePostJump`).
Between events the state only decays, eq. (1.3.38, `eq.update_of_exponential_intensity`):
$$Z(T_{n+1}) = \big(Z(T_n) + \mathbf e_{E_n}\big)\,e^{-\beta(T_{n+1} - T_n)}.$$

**F5. The exact scheme.**
Between events $\lambda_{\mathfrak g}(t) = \bar\mu + D\,e^{-\beta(t - T_n)}$,
with $D = \lambda_{\mathfrak g}(T_n+) - \bar\mu = (\mathbf 1^\top A)\,Z(T_n+) \geq 0$,
eq. (1.3.39, `eq.totalIntensityBetweenEvents`).
By Theorem 1.3.43 (`thm.dassiosZhao`), with $U_1, U_2$ independent and uniform on $(0,1)$,
$$\tau_1 = -\frac{\log U_1}{\bar\mu},
\qquad
\tau_2 = -\frac{1}{\beta}\log\Big(1 + \frac{\beta\log U_2}{D}\Big),$$
eq. (1.3.41, `eq.dzDraws`), with $\tau_2 = +\infty$ when $D = 0$ or $1 + \beta\log U_2/D \leq 0$;
the wait is $\tau_1 \wedge \tau_2$,
and the type is drawn at the arrival, before the jump,
with probabilities $\lambda_e(T_{n+1})/\lambda_{\mathfrak g}(T_{n+1})$, eq. (1.3.42, `eq.dzType`).
Algorithm 1.3.2 (`algo.dassiosZhao`) orders the steps:
the wait, the decay of the state to the arrival, the type, the jump.

## 6.1 · `question.thinningBoundAfterTheJump`

The self-exciting pair, $\mu = (0.25, 0.25)$,
$A = \begin{pmatrix} 2.8 & 0.2 \\ 0.2 & 2.8 \end{pmatrix}$, $\beta = 4$,
over $2\times10^4$ s, where the standard error of the rate is $0.04$.

**Answer: (D).** A prints a number close to 2, and B a number clearly below 2.

**The total rate.**
$\Gamma = A/4$ has both column sums $(2.8 + 0.2)/4 = 0.75$ (F1), so $\rho = 0.75$ (F2),
and with $\bar\mu = 0.5$ (F3)
$$\bar\lambda = \frac{\bar\mu}{1-\rho} = \frac{0.5}{0.25} = 2.$$

In [ ]:
rho = (2.8 + 0.2) / 4
mu_bar = 0.25 + 0.25
rho, mu_bar / (1 - rho)

**The bound.**
Thinning, Algorithm 1.3.1 (`algo.ogata`), draws candidates at the rate of a bound $b$
and keeps each with probability $\lambda_{\mathfrak g}/b$.
It is exact when $b$ dominates $\lambda_{\mathfrak g}$ up to the next accepted candidate.
The paragraph on thinning that precedes the algorithm, in Section 1.3.5 (`sec.simulation`), supplies the bound:
$A \geq 0$ makes $\lambda_{\mathfrak g}$ non-increasing between events,
so the post-jump value $\lambda_{\mathfrak g}(T_n+)$ dominates it up to $T_{n+1}$;
the pre-jump value is smaller by $(\mathbf 1^\top A)_{E_n}$, here 3, and does not.

- **A** sets `bound = mu_bar + jumps @ state` after the jump, that is $b = \lambda_{\mathfrak g}(T_n+)$,
  and keeps it through the rejections.
  A bound never tightened costs rejections and nothing else:
  A is exact, and prints a number within a few standard errors of 2.
- **B** sets `bound = intensity` at every candidate.
  After a rejection this is $\lambda_{\mathfrak g}$ at the candidate,
  which dominates what follows, as in Algorithm 1.3.1 (`algo.ogata`).
  After an acceptance it is $\lambda_{\mathfrak g}(T_n)$, read before the jump,
  3 below $\lambda_{\mathfrak g}(T_n+)$.
  While $\lambda_{\mathfrak g} > b$ the test `rng.random() * bound <= intensity` always passes,
  so events arrive at the rate $b$ and not at the rate $\lambda_{\mathfrak g}$:
  the excitation of each accepted event is never seen in full.
  From a quiet state, $\lambda_{\mathfrak g} \approx \bar\mu = 0.5$,
  an event raises $\lambda_{\mathfrak g}$ to 3.5 while B keeps proposing at 0.5,
  and its next candidate comes after a wait $W$ of mean 2 s,
  over which the excitation keeps on average $\mathbb E[e^{-4W}] = 0.5/(0.5 + 4) = 1/9$ of itself.
  B's bound stays near $\bar\mu$, and B prints a number clearly below 2.
  The verification notebook prints 0.72.

**The other choices.**
- (A) Both close to 2: would hold only if the pre-jump value were a bound.
- (B) A below, B close: exchanges the two alternatives. A's bound is loose, not low.
- (C) B clearly above: a bound below the intensity loses events and cannot add any,
  since every event of B is a candidate kept with probability at most one.

## 6.2 · `question.recordedStatesAliasTheJump`

A buy, a sell and a buy at $0.5$, $1.0$ and $1.5$ s;
$\beta = 2\log 2$, so that $e^{-\beta \cdot 0.5} = 1/2$.

**Answer: (C).**
A prints `[[1.0, 0.0], [0.5, 1.0], [1.25, 0.5]]`,
and B prints `[[1.25, 0.5], [1.25, 0.5], [1.25, 0.5]]`.

**What should be recorded.**
By eq. (1.3.38, `eq.update_of_exponential_intensity`) (F4) the pre-jump states are
$$Z(0.5) = (0, 0),
\qquad Z(1.0) = \tfrac12\big((0,0) + \mathbf e_0\big) = (0.5, 0),
\qquad Z(1.5) = \tfrac12\big((0.5,0) + \mathbf e_1\big) = (0.25, 0.5),$$
and the post-jump states $Z(T_n+) = Z(T_n) + \mathbf e_{E_n}$ are
$(1, 0)$, $(0.5, 1)$ and $(1.25, 0.5)$.
Remark 1.3.41 (`remark.prePostJump`) keeps the two apart.

**What is recorded.**
`record.append(state)` stores a reference to the array, not a copy of it.
- In A, `state = state * ...` binds `state` to a new array at every event.
  The record holds that array,
  and `state[event_type] += 1.0` then adds the jump to it, in place:
  each entry is the post-jump state of its own event.
- In B, `state *= ...` decays, in place, the one array made by `np.zeros(2)`.
  The record holds three references to it,
  and prints its final value, $(1.25, 0.5)$, three times.

Recording `state.copy()` gives the pre-jump states.

In [ ]:
half = exp(-2 * log(2) * 0.5)
half, (half * (0 + 1), half * 0), (half * 0.5, half * (0 + 1))

**The other choices.**
- (A) A records the pre-jump states:
  misses that the jump is added, in place, to the array A has just recorded.
- (B) Both record the pre-jump states: misses the aliasing in both.
- (D) Both record the post-jump states: misses that `*=` keeps one array through the loop.

## 6.3 · `question.intensityByRecursionAgainstSum`

$\lambda_{\mathfrak g}(T_n)$ at every event, by the sum over the past (A) and by the update (B),
timed on the first $n$ events of a path, $n = 1000, \dots, 16000$.

**Answer: (A).**
The time per event of B stays about the same and that of A grows with $n$:
B is faster at every $n$, by a factor that grows with $n$.

**The cost.**
A evaluates eq. (1.3.11, `eq.decayedCounts`) directly:
at event $n$ the weights `np.exp(-params.decay * (now - times[:n]))` are $n$ terms,
one per earlier event,
so the path costs $\sum_{k<n} k = n(n-1)/2$ terms, $O(n^2)$,
and the time per event grows linearly in $n$.
B carries the state by eq. (1.3.38, `eq.update_of_exponential_intensity`),
a fixed number of operations per event, $O(1)$.
The sentence after eq. (1.3.38, `eq.update_of_exponential_intensity`) states both costs.

Vectorising A's sum lowers its constant and not its growth.
At small $n$ A's time is dominated by the fixed cost of its numpy calls,
and the growth shows as $n$ increases.

**The other choices.**
- (B) Both constant per event: misses that A's sum runs over the whole past.
- (C) A faster, its body being one vectorised expression:
  one vectorised expression over $n$ terms is still $n$ terms.
- (D) Both grow, since each loops over the events:
  the outer loop is common to both and gives the factor $n$ of the path,
  not a growth of the time per event; only the body of A grows.

## 6.4 · `question.exactWaitOnGivenUniforms`

The asymmetric pair, $\mu = (0.2, 0.6)$,
$A = \begin{pmatrix} 1.2 & 4.8 \\ 0.4 & 0.8 \end{pmatrix}$, $\beta = 4$,
from $Z(0+) = (1.5, 1.0)$.
$U_1 = e^{-0.4}$; $U_2 = e^{-1}$ in the first case and $e^{-3}$ in the second;
the third uniform, $0.75$, goes to the type draw.

**Answer: (D).** `0.173`, then `0.5`.

**The quantities of F5.**
$\bar\mu = 0.8$, and the column sums of $A$ are $\mathbf 1^\top A = (1.6, 5.6)$, so
$$D = (\mathbf 1^\top A)\,Z(0+) = 1.6 \cdot 1.5 + 5.6 \cdot 1.0 = 8,$$
the excess of $\lambda_{\mathfrak g}(0+) = 8.8$ over $\bar\mu$, which is `excess` in the code.
By eq. (1.3.41, `eq.dzDraws`), $-\log U_1 = 0.4$ gives $\tau_1 = 0.4/0.8 = 0.5$ in both cases.

**First case, $\log U_2 = -1$.**
$1 + \beta\log U_2/D = 1 - 4/8 = 1/2 > 0$, so
$$\tau_2 = -\tfrac14\log\tfrac12 = \tfrac14\log 2 \approx 0.173,$$
and the wait is $\tau_1 \wedge \tau_2 = 0.173$.

**Second case, $\log U_2 = -3$.**
$1 - 12/8 = -1/2 \leq 0$, so $\tau_2 = +\infty$ by the convention of Theorem 1.3.43 (`thm.dassiosZhao`):
the excitation clock never fires, Remark 1.3.44 (`remark.defectiveClock`).
The wait is $\tau_1 = 0.5$.

`from_state` sets the clock to zero, so the printed `time` is the wait.

In [ ]:
mu_bar = 0.2 + 0.6
D = (1.2 + 0.4) * 1.5 + (4.8 + 0.8) * 1.0
tau_1 = 0.4 / mu_bar
remaining_1 = 1 + 4 * (-1) / D
remaining_2 = 1 + 4 * (-3) / D
D, tau_1, remaining_1, -log(remaining_1) / 4, remaining_2

**The other choices.**
- (A) `0.5`, then `0.5`:
  the column sums of $\Gamma = A/4$, $(0.4, 1.4)$, taken for those of $A$ (F1).
  Then $D = 0.4 \cdot 1.5 + 1.4 \cdot 1.0 = 2$, and $1 - 4/2 = -1$ and $1 - 12/2 = -5$ are both negative:
  $\tau_2 = +\infty$ twice, and the wait is $\tau_1 = 0.5$ twice.
- (B) `0.5`, then `inf`: the later of the two clocks taken for the earlier,
  $\tau_1 \vee \tau_2$ for $\tau_1 \wedge \tau_2$: $0.5 \vee 0.173 = 0.5$, then $0.5 \vee \infty = \infty$.
- (C) `0.173`, then `0.32`: $-\log U_1$ multiplied by $\bar\mu$ rather than divided by it,
  $0.4 \cdot 0.8 = 0.32$; in the first case the excitation clock still fires first.
- (E) `0.173`, then `inf`: the infinite $\tau_2$ returned as the wait, the minimum with $\tau_1$ forgotten.

In [ ]:
D_gamma = 0.4 * 1.5 + 1.4 * 1.0
D_gamma, 1 + 4 * (-1) / D_gamma, 0.4 * mu_bar

## 6.5 · `question.typeDrawnBeforeTheDecay`

The state and the uniforms of the first case of 6.4.
`DrawFirst` draws the type before `_advance`.

**Answer: (E).** `1 [0.75, 1.5]`, then `0 [1.25, 0.5]`.

**The package.**
The wait of 6.4 is $\tau_2 = \tfrac14\log 2$,
and `_advance` multiplies the state by $e^{-\beta\tau_2} = 1 + \beta\log U_2/D = 1/2$:
$Z(T_1) = (0.75, 0.5)$.
The type is drawn on the intensities at the arrival, before the jump, eq. (1.3.42, `eq.dzType`),
with $\lambda = \mu + AZ$, eq. (1.3.13, `eq.intensityFromState`):
$$\lambda(T_1) =
\begin{pmatrix} 0.2 + 1.2 \cdot 0.75 + 4.8 \cdot 0.5 \\ 0.6 + 0.4 \cdot 0.75 + 0.8 \cdot 0.5 \end{pmatrix}
= \begin{pmatrix} 3.5 \\ 1.3 \end{pmatrix}.$$
The cumulative sums are $(3.5, 4.8)$ and the threshold is $0.75 \cdot 4.8 = 3.6$.
`np.searchsorted` returns the first index at which the cumulative sum reaches 3.6, which is 1:
a market sell, and the jump gives $Z(T_1+) = (0.75, 1.5)$.

In [ ]:
z = (0.75, 0.5)
lam = (0.2 + 1.2 * z[0] + 4.8 * z[1], 0.6 + 0.4 * z[0] + 0.8 * z[1])
lam, lam[0] + lam[1], 0.75 * (lam[0] + lam[1])

**`DrawFirst`.**
It draws on the intensities at $Z(0+) = (1.5, 1.0)$, those left by the last event:
$$\mu + A\,Z(0+) = (0.2 + 1.8 + 4.8,\; 0.6 + 0.6 + 0.8) = (6.8, 2.0).$$
The cumulative sums are $(6.8, 8.8)$ and the threshold is $0.75 \cdot 8.8 = 6.6 \leq 6.8$: a market buy.
The jump gives $(2.5, 1.0)$,
and `_advance` then halves the state with its own jump in it, to $(1.25, 0.5)$.
The wait is unchanged, since `excess` is computed before the draw.

Algorithm 1.3.2 (`algo.dassiosZhao`) orders the steps as the package does:
the decay to the arrival, the draw on $\lambda(T_{n+1})$, the jump.

In [ ]:
z = (1.5, 1.0)
lam = (0.2 + 1.2 * z[0] + 4.8 * z[1], 0.6 + 0.4 * z[0] + 0.8 * z[1])
lam, lam[0] + lam[1], 0.75 * (lam[0] + lam[1])

**The other choices.**
- (A) `1 [0.75, 1.5]` on both lines:
  reads `DrawFirst` as the package, missing that its draw comes before the decay.
- (B) `0 [1.75, 0.5]` on the second line:
  the buy of `DrawFirst` with its jump added after the halving, $(0.75 + 1, 0.5)$;
  the code adds the jump first, and halves it with the state.
- (C) `0 [2.5, 1.0]` on the second line: the state of `DrawFirst` read before `_advance`.
- (D) `0 [1.75, 0.5]` on the first line:
  the threshold 3.6 taken for falling among the buys, whose cumulative sum, 3.5, it exceeds.

## 6.6 · `question.clippedExcitationClockAddsEvents`

$\mu = (0.5, 0.5)$, $A = \begin{pmatrix} 15 & 10 \\ 10 & 15 \end{pmatrix}$, $\beta = 50$,
over $2\times10^4$ s, where the standard error of the rate is $0.02$.
`Clipped` replaces $1 + \beta\log U_2/D$ by its maximum with $10^{-12}$.

**Answer: (B).**
`ExponentialHawkes` prints a number close to 2, and `Clipped` a number clearly above 2.

**The total rate.**
The column sums of $A$ are 25, so those of $\Gamma = A/50$ are $0.5 = \rho$ (F1, F2),
and $\bar\lambda = \bar\mu/(1-\rho) = 1/0.5 = 2$ (F3).

**The defective clock.**
By the proof of Theorem 1.3.43 (`thm.dassiosZhao`), $\tau_2 = +\infty$ exactly when $-\log U_2 \geq D/\beta$,
an event of probability $e^{-D/\beta}$:
the excited factor of eq. (1.3.40, `eq.dzSurvival`) carries the finite mass $D/\beta$,
the expected number of direct offspring the state still owes, Remark 1.3.44 (`remark.defectiveClock`).
Just after an event $D \geq 25$, so this probability is at most $e^{-1/2} \approx 0.61$;
after a quiet spell $D$ is near 0 and the probability near 1.
The package then takes the wait $\tau_1$, and the next event is an immigrant.

**The clipped clock.**
`Clipped` fires every such clock at $-\log(10^{-12})/\beta \approx 27.6/50 \approx 0.553$ s.
It fires before the baseline clock whenever $\tau_1 > 0.553$,
an event of probability $e^{-\bar\mu \cdot 0.553} = e^{-0.553} \approx 0.58$.
Each such firing is an event the model does not have, and it raises the intensity,
so it calls forth further events.

In [ ]:
rho = (15 + 10) / 50
clip = -log(1e-12) / 50
rho, 1 / (1 - rho), exp(-25 / 50), clip, exp(-1.0 * clip)

**The other choices.**
- (A) Both close to 2: treats $\tau_2 = +\infty$ as a numerical edge case,
  which Remark 1.3.44 (`remark.defectiveClock`) says it is not: clipping it biases the immigrant share.
- (C) `Clipped` below 2: the clip only shortens waits, so it can only add events.
- (D) The package below 2, its clock failing to fire:
  an infinite $\tau_2$ loses no event; the wait is then $\tau_1$,
  and $\tau_1 \wedge \tau_2$ has the law eq. (1.3.40, `eq.dzSurvival`).

## 6.7 · `question.selfAndCrossExcitingShareTheirTimes`

The self-exciting pair, $A = \begin{pmatrix} 2.8 & 0.2 \\ 0.2 & 2.8 \end{pmatrix}$,
and the cross-exciting pair, $A = \begin{pmatrix} 0.2 & 2.8 \\ 2.8 & 0.2 \end{pmatrix}$,
with $\mu = (0.25, 0.25)$ and $\beta = 4$, simulated on seed 3 over 100 s.

**Answer: (C).** `True False`.

**The times.**
Both matrices have column sums $(3, 3)$,
so under both $\lambda_{\mathfrak g} = 0.5 + 3\,\mathbf 1^\top Z$ (F4).
By eq. (1.3.11, `eq.decayedCounts`), $\mathbf 1^\top Z(t) = \sum_{T_n < t} e^{-\beta(t - T_n)}$
is a function of the times of the past events and not of their types.
So $D = 3\,\mathbf 1^\top Z(T_n+)$, and the waits of eq. (1.3.41, `eq.dzDraws`),
are the same functions of the past times under both kernels.
Each step draws two uniforms for the wait and one for the type, under both,
so the two generators of seed 3 stay in step:
the same uniforms give the same waits,
and the times agree up to floating-point rounding,
since `_total_jump @ decayed_counts` sums different components on the two paths.
`np.allclose` prints `True`.

**The types.**
The type is drawn on the full intensities $\mu + AZ$, eq. (1.3.42, `eq.dzType`), where the two matrices differ.
By Remark 1.3.45 (`remark.typeDraw`) the wait sees $A$ only through $\mathbf 1^\top A$,
and the type draw is where the full matrix re-enters.
Over about 200 events ($\bar\lambda = 2$) the two type sequences differ: `False`.

**The other choices.**
- (A) `True True`: misses that the type draw sees the full matrix.
- (B) `False False`: takes the waits to see the matrix beyond its column sums.
- (D) `ValueError`: needs paths of different lengths; with the same times they have the same length.

## 6.8 · `question.eventsHoldTheCrossingEvent`

A simulator of the one-way pair, seed 11, run to 10 s and then to 40 s,
against a single run to 40 s.

**Answer: (A).** `True 0`, then `False 1`.

`self.step()` draws the next event and, in drawing it,
moves the clock, decays the state, adds the jump (Algorithm 1.3.2 (`algo.dassiosZhao`))
and consumes three uniforms.
The first event past 10 s is drawn in this way before its time is compared with the horizon.
- The package stores it in `self._crossing` and returns.
  The second call yields it first, since it lies before 40 s,
  and continues with `step()` from the state after it.
  The two stretches are the single run: `True 0`.
- `Dropping` returns without storing it.
  Its second call continues with `step()` from the same clock and state,
  which already include the dropped event,
  so every later event is the one the single run draws, and only the dropped one is missing:
  `False 1`.

In both, the second call continues from where the first stopped, not from time zero.

**The other choices.**
- (B) `True 0` on both lines: takes the dropped event for undrawn,
  as though the clock and the state had not moved.
- (C) `False 1` on both lines: misses that the package holds the crossing event.
- (D) `False` and a negative number: reads the second call as restarting from time zero,
  which would give the two stretches more events than the single run.

## 6.9 · `question.seedsOfAnOrderFlowMonteCarlo`

The weakly exciting pair, $\bar\mu = 1.8$ and $\bar\lambda = 2$; paths of 10 s.
The quoted line keeps a `Generator` as given and builds a fresh one from anything else.

**Answer: (B).** `True False False`.

- `path(5) == path(5)`: each call builds `np.random.default_rng(5)`,
  so both paths are drawn from the start of one stream: `True`.
- `path(rng) == path(rng)`: the two simulators share one generator,
  and the second reads the part of its stream after the part the first consumed.
  Two different paths, both non-empty:
  a path is empty with probability $e^{-\bar\mu \cdot 10} = e^{-18}$.
  `False`.
- `path(5) == path(rng)`: the arguments of `print` are evaluated from left to right,
  so this `path(rng)` is the third use of `rng`,
  which has moved past the numbers with which `default_rng(5)` starts: `False`.

Had `path(rng)` been called first, it would have equalled `path(5)`,
since `rng` is `default_rng(5)`.
Paths that are independent and reproducible are drawn from one generator,
seeded once and passed to every simulator.

In [ ]:
exp(-1.8 * 10)

**The other choices.**
- (A) `True False True`: takes the third `path(rng)` for the start of the stream of `default_rng(5)`,
  forgetting the two calls before it.
- (C) `True True True`: takes the simulator to copy the generator it is given.
- (D) `False False False`: takes an integer seed to draw fresh entropy at every call.

## 6.10 · `question.transposedKernelChangesTheRate`

The records of the asymmetric pair, kernel $(1.2, 4.8, 0.4, 0.8)$ in the order
(Component, Cause) $= (0,0), (0,1), (1,0), (1,1)$;
$\mu = (0.2, 0.6)$, $\beta = 4$; over $2\times10^4$ s, with standard errors below $0.04$.

**Answer: (B).** A number close to 3, then a number close to 2.

**The two matrices.**
`reshape(2, 2)` fills rows first, so row $e$ holds the records with `Component` $e$:
it is $A$, row excited and column exciting, as Remark 1.3.7 (`remark.indexOrder`) requires.
Its transpose exchanges $A_{0,1} = 4.8$ and $A_{1,0} = 0.4$.

**Stability.**
$\Gamma = A/4 = \begin{pmatrix} 0.3 & 1.2 \\ 0.1 & 0.2 \end{pmatrix}$
has trace $0.5$ and determinant $0.06 - 0.12 = -0.06$,
so its eigenvalues solve $x^2 - 0.5x - 0.06 = 0$: they are $0.6$ and $-0.1$, and $\rho = 0.6$.
$\Gamma^\top$ has the same eigenvalues.
Both satisfy the condition $\rho < 1$ of Theorem 1.3.19 (`thm.stationarity`), and `HawkesParams` raises nothing.

In [ ]:
trace, determinant = 0.3 + 0.2, 0.3 * 0.2 - 1.2 * 0.1
(trace + sqrt(trace**2 - 4 * determinant)) / 2, (trace - sqrt(trace**2 - 4 * determinant)) / 2

**The rates.**
By eq. (1.3.18, `eq.stationaryIntensity`), with $\det(I - \Gamma) = 0.7 \cdot 0.8 - 1.2 \cdot 0.1 = 0.44$,
$$\lambda^* = \frac{1}{0.44}\begin{pmatrix} 0.8 & 1.2 \\ 0.1 & 0.7 \end{pmatrix}
\begin{pmatrix} 0.2 \\ 0.6 \end{pmatrix} = \begin{pmatrix} 2 \\ 1 \end{pmatrix},
\qquad
(I - \Gamma^\top)^{-1}\mu = \frac{1}{0.44}\begin{pmatrix} 0.8 & 0.1 \\ 1.2 & 0.7 \end{pmatrix}
\begin{pmatrix} 0.2 \\ 0.6 \end{pmatrix} = \begin{pmatrix} 0.5 \\ 1.5 \end{pmatrix}.$$
The total rates are 3 and 2: the two prints lie some 25 standard errors apart.

The column sums of $\Gamma$, $0.4$ and $1.4$, are not constant, and F3's shortcut does not apply to the first matrix.
For the transpose $\mu$ is a right Perron vector, $\Gamma^\top\mu = 0.6\,\mu$,
so $(I - \Gamma^\top)^{-1}\mu = \mu/(1-\rho)$,
the second of the sufficient conditions stated after Corollary 1.3.25 (`corol.constantClusterSize`),
and its rate is $\bar\mu/(1-\rho) = 0.8/0.4 = 2$.

Remark 1.3.7 (`remark.indexOrder`): a transposed kernel passes every stability test,
and gives the right total rate when $\mu$ is solved from a target $\lambda^*$.
At a fixed $\mu$ the rate changes.

In [ ]:
det = 0.7 * 0.8 - 1.2 * 0.1
lam = ((0.8 * 0.2 + 1.2 * 0.6) / det, (0.1 * 0.2 + 0.7 * 0.6) / det)
lam_t = ((0.8 * 0.2 + 0.1 * 0.6) / det, (1.2 * 0.2 + 0.7 * 0.6) / det)
det, lam, sum(lam), lam_t, sum(lam_t)

**The other choices.**
- (A) Close to 3 twice: holds when $\mu$ is solved from a target $\lambda^*$, Remark 1.3.7 (`remark.indexOrder`);
  here $\mu$ is fixed.
- (C) Close to 2, then close to 3: reads `reshape` as filling columns first.
- (D) Close to 3, then `ValueError`: a matrix and its transpose share their spectral radius.

# Section 7 · Necessary conditions and goodness of fit

**G1. Compensator and intensity.**
The compensator $\Lambda$ of $N$ starts at 0, is of finite variation, predictable and right-continuous,
and makes $N - \Lambda$ a local martingale, Definition 1.3.2 (`def.compensator`).
It is equivalently characterised as the predictable non-decreasing process such that,
for every non-negative predictable $Y$,
$$\mathbb E\Big[\int Y\,dN\Big] = \mathbb E\Big[\int Y\,d\Lambda\Big],$$
eq. (1.3.5, `eq.compensators_and_predict_projections`),
and $d\Lambda = \lambda\,dt$ when $N$ has an intensity.

**G2. The closed form.**
For the exponential kernel started at $N(0) = 0$, Proposition 1.3.12 (`prop.compensatorClosedForm`) gives
$$\Lambda_e(T) = \mu_e T + \sum_{e'}\frac{A_{e,e'}}{\beta}\big(N_{e'}(T-) - Z_{e'}(T)\big),$$
eq. (1.3.15, `eq.compensatorClosedForm`): the count read before $T$, the state at it.
The map $T \mapsto N_{e'}(T-) - Z_{e'}(T)$ is continuous, the two jumps cancelling;
the post-jump reading $N_{e'}(T) - Z_{e'}(T)$ overstates $\Lambda_e$ by $A_{e,e'}/\beta$
at each arrival of type $e'$, Remark 1.3.13 (`remark.readPreJump`).

**G3. The unit of time.**
For $c > 0$, the process whose arrival times are $c\,T_j$
is the exponential-kernel process with $\mu/c$, $A/c$ and $\beta/c$,
and its compensator at $c\,T$ is $\Lambda(T)$, Remark 1.3.11 (`remark.timeRescaling`).

**G4. Meyer's theorem.**
By Theorem 1.3.5 (`thm.meyer1971`), if
i. no two components jump together, eq. (1.3.1, `eq.noCommonJumps`);
ii. $\Lambda$ is continuous;
iii. $\Lambda_e(t) \to \infty$ for every $e$,
then the residuals
$$\tau^j_e = \Lambda_e(T^e_j) - \Lambda_e(T^e_{j-1}),$$
eq. (1.3.7, `eq.time-changed_intertimes`), are independent $\mathrm{Exp}(1)$, jointly over $j$ and $e$.
The test of goodness of fit of Section 1.3.5 (`sec.simulation`) compares them with $\mathrm{Exp}(1)$.
A comparison of each margin sees neither the serial nor the joint independence, Remark 1.3.46 (`remark.marginsOnly`).

**G5. The self-exciting pair.**
$\mu = (0.25, 0.25)$, $A = \begin{pmatrix} 2.8 & 0.2 \\ 0.2 & 2.8 \end{pmatrix}$, $\beta = 4$:
the column sums are 3 for $A$ and $0.75 = \rho$ for $\Gamma$,
$\lambda^* = (1, 1)$ and $\bar\lambda = 2$ (F1–F3).

**G6. Kolmogorov–Smirnov.**
The statistic is $D = \sup_x |F_n(x) - F(x)|$.
Two facts from outside the notes:
$D$ is unchanged when the sample and the law are scaled together, $x \mapsto cx$;
and under the null, for large $n$, $D$ is of order $0.87/\sqrt n$
and exceeds $1.63/\sqrt n$ with probability $0.01$.

## 7.1 · `question.palmMeanOfTheTotalIntensity`

The self-exciting pair (G5), one path over $4\times10^4$ s.
`intensities_at_events` records $\lambda(T_n)$ after the decay and before the jump.

**Answer: (B).** `2.0 6.5 9.5`.

**The rate.** $N(T)/T \to \bar\lambda = 2$ (F3).

**The mean before the jump.**
`before[n]` is $\lambda_{\mathfrak g}(T_n)$, the predictable value.
Apply eq. (1.3.5, `eq.compensators_and_predict_projections`) to the ground process
with $Y = \lambda_{\mathfrak g}\,\mathbf 1_{(0,T]}$, non-negative and predictable:
$$\mathbb E\Big[\sum_{T_n \leq T}\lambda_{\mathfrak g}(T_n)\Big]
= \mathbb E\Big[\int_0^T \lambda_{\mathfrak g}\,dN_{\mathfrak g}\Big]
= \mathbb E\Big[\int_0^T \lambda_{\mathfrak g}(t)^2\,dt\Big]
\approx T\,\mathbb E\big[\lambda_{\mathfrak g}^2\big],$$
the last at stationarity.
Divided by the count, about $\bar\lambda T$, the mean over the events is
$$\frac{\mathbb E[\lambda_{\mathfrak g}^2]}{\bar\lambda}
= \frac{\mathrm{Var}(\lambda_{\mathfrak g}) + \bar\lambda^2}{\bar\lambda},$$
with $\mathbb E[\lambda_{\mathfrak g}] = \mathbf 1^\top\lambda^* = \bar\lambda$ by eq. (1.3.18, `eq.stationaryIntensity`).
The events fall where the intensity is high, and over-sample it.

**The variance.**
$\lambda_{\mathfrak g} = 0.5 + 3\,\mathbf 1^\top Z$ and $A$ has constant column sums,
so eq. (1.3.29, `eq.scalarVariance`) applies:
$$\mathrm{Var}(\mathbf 1^\top Z) = \frac{\bar\lambda}{2\beta(1-\rho)} = \frac{2}{2 \cdot 4 \cdot 0.25} = 1,
\qquad
\mathrm{Var}(\lambda_{\mathfrak g}) = 9\,\mathrm{Var}(\mathbf 1^\top Z) = 9.$$
Remark 1.3.32 (`remark.scalarVariance`) gives the same, $\beta\rho^2\bar\lambda/(2(1-\rho)) = 9$.
The mean before the jump is $(9 + 4)/2 = 6.5$.

**After the jump.**
`after = before + params.excitation.sum(axis=0)[types]` adds the column sum of $A$
of the event's type, 3 for either: $6.5 + 3 = 9.5$.

In [ ]:
rate, rho, beta = 2.0, 0.75, 4.0
var_sum_z = rate / (2 * beta * (1 - rho))
var_total = 9 * var_sum_z
var_sum_z, var_total, beta * rho**2 * rate / (2 * (1 - rho)), (var_total + rate**2) / rate

**The other choices.**
- (A) `2.0 2.0 5.0`: the time average $\mathbb E[\lambda_{\mathfrak g}] = 2$ taken for the average over the events.
- (C) `2.0 2.5 5.5`: $\mathrm{Var}(\mathbf 1^\top Z) = 1$ taken for $\mathrm{Var}(\lambda_{\mathfrak g})$, $(1 + 4)/2$;
  Remark 1.3.32 (`remark.scalarVariance`): the formula concerns the total decayed count, not the total intensity.
- (D) `2.0 4.5 7.5`: $\mathrm{Var}(\lambda_{\mathfrak g})/\bar\lambda$, the squared mean left out.

## 7.2 · `question.exponentialReferenceOfAPoissonFlow`

3000 gaps of a Poisson flow of rate 3, tested against five reference laws.
`stats.expon(loc, scale)` is the law of $\mathtt{loc} + \mathtt{scale}\cdot X$, $X \sim \mathrm{Exp}(1)$.

**Answer: (D).** `[0, 2]`.

The compensator is $\Lambda(t) = 3t$,
so by Theorem 1.3.5 (`thm.meyer1971`) the residuals $3\,\Delta t$ are independent $\mathrm{Exp}(1)$.
- Reference 2, `(3 * gaps, stats.expon().cdf)`, is the residuals against $\mathrm{Exp}(1)$, the test of G4.
  It passes, with probability $0.99$ at the level $0.01$.
- Reference 0, `(gaps, stats.expon(scale=1 / 3).cdf)`, is the raw gaps against $F(x) = 1 - e^{-3x}$:
  reference 2 with sample and law scaled together by $1/3$.
  The same statistic (G6), the same p-value: it passes.
- Reference 1, `stats.expon(3)`: the first positional argument is `loc`,
  so the law is $3 + \mathrm{Exp}(1)$, and $F = 0$ below 3.
  A gap exceeds 3 with probability $e^{-9}$, so $D \approx 1$.
- Reference 3, `scale=3`: a mean of 3, a rate of $1/3$.
  $D = \max_x\big(e^{-x/3} - e^{-3x}\big) \approx 0.68$, at $x = \tfrac38\log 9$.
- Reference 4, `stats.expon()` on the raw gaps:
  $D = \max_x\big(e^{-x} - e^{-3x}\big) = 3^{-1/2} - 3^{-3/2} = 2/(3\sqrt3) \approx 0.385$,
  at $x = \tfrac12\log 3$.

The last three distances are far above the 1% critical value $1.63/\sqrt{3000} \approx 0.03$ (G6),
and all three are rejected.

In [ ]:
x_3, x_4 = 3 / 8 * log(9), log(3) / 2
exp(-9), exp(-x_3 / 3) - exp(-3 * x_3), exp(-x_4) - exp(-3 * x_4), 1.63 / sqrt(3000)

**The other choices.**
- (A) `[0, 1, 2]`: reads `stats.expon(3)` as a rate of 3.
- (B) `[2, 3]`: reads `scale` as a rate, so that `scale=3` would fit and `scale=1/3` would not.
- (C) `[0, 2, 4]`: extends the scale invariance of G6 to a law not scaled with the sample.
- (E) `[2]`: misses that reference 0 is reference 2 rescaled.

## 7.3 · `question.compensatorInMilliseconds`

A path of the self-exciting pair (G5) over 4000 s, about 4000 buys a second apart,
read through four pairs of a clock and a specification.

**Answer: (A).** 1, 250, 1, 0.25.

With `prepend=0.0` the residuals of the buys sum to $\Lambda_0$ at the last buy,
so their mean is $\Lambda_0(T^0_{N_0})/N_0$.
On the clock of seconds, by eq. (1.3.15, `eq.compensatorClosedForm`),
$$\Lambda_0(T) = 0.25\,T + 0.7\,\big(N_0 - Z_0\big) + 0.05\,\big(N_1 - Z_1\big)
\approx (0.25 + 0.7 + 0.05)\,T = T,$$
with $N_0 \approx N_1 \approx T$ from $\lambda^* = (1, 1)$ and $Z$ of order one:
a mean of $0.25$ per buy from the immigrants and $0.75$ from the excitation.

- **Line 1**, seconds and the true specification:
  the residuals are $\mathrm{Exp}(1)$ by Theorem 1.3.5 (`thm.meyer1971`), with mean about 1.
- **Line 3**, milliseconds and $(\mu, A, \beta)/1000$:
  by Remark 1.3.11 (`remark.timeRescaling`) with $c = 1000$ this is the same process,
  and its compensator at $1000\,T$ is $\Lambda(T)$. The same residuals, with mean about 1.
- **Line 2**, milliseconds and the specification per second.
  A decay of 4 per millisecond: consecutive events lie about 500 ms apart,
  where $Z$ decays by $e^{-2000}$, so $Z \approx 0$ at nearly every event.
  With $t$ in seconds and the clock reading $1000\,t$,
  $$\Lambda_0 \approx 0.25 \cdot 1000\,t + \frac{2.8}{4}N_0 + \frac{0.2}{4}N_1 = 250\,t + 0.7\,N_0 + 0.05\,N_1,$$
  a mean of about $250 + 0.75 = 250.75$.
  The baseline term is a thousand times too large, and the excitation keeps its $0.75$.
- **Line 4**, milliseconds, $\mu/1000$ and $A/1000$, $\beta = 4$ per millisecond: again $Z \approx 0$, and
  $$\Lambda_0 \approx 0.25\,t + 0.0007\,N_0 + 0.00005\,N_1,$$
  a mean of about $0.25075$.
  The baseline term is right, and the excitation a thousand times too small.

In [ ]:
mu_0, a_00, a_01, beta = 0.25, 2.8, 0.2, 4.0
line_2 = 1000 * mu_0 + a_00 / beta + a_01 / beta
line_4 = mu_0 + (a_00 + a_01) / 1000 / beta
exp(-beta * 500), line_2, line_4

**The other choices.**
- (B) 1, 1, 1, 1: reads the compensator as free of the unit of time.
- (C) 1, 250, 1, 1: rescales $\mu$ and $A$ and leaves $\beta$ per second;
  by Remark 1.3.11 (`remark.timeRescaling`) the three are rates and scale together.
- (D) 1, 1000, 1, 0.25: takes the mean gap of the buys in milliseconds for the second line;
  the compensator multiplies the time by $\mu_0 = 0.25$, and the excitation adds $0.75$ per buy.

## 7.4 · `question.compensatorReadAfterTheJump`

The self-exciting pair (G5), one path over 2000 s.
B records after both updates, C after the count and before the state.

**Answer: (C).** `[] []`, then `[0] [0.7]`.

The package records row $k$ after the decay and before both updates,
$\mu T_k + A\big(N(T_k-) - Z(T_k)\big)/\beta$, which is eq. (1.3.15, `eq.compensatorClosedForm`) at $T_k$.
- **B** reads $N(T_k) - Z(T_k+) = \big(N(T_k-) + \mathbf e_{E_k}\big) - \big(Z(T_k) + \mathbf e_{E_k}\big)
  = N(T_k-) - Z(T_k)$.
  The two jumps cancel, Remark 1.3.13 (`remark.readPreJump`):
  the same numbers, to rounding, and no residual moves.
- **C** reads $N(T_k) - Z(T_k) = N(T_k-) - Z(T_k) + \mathbf e_{E_k}$,
  the post-jump reading of Remark 1.3.13 (`remark.readPreJump`).
  Row $k$ of $\Lambda_0$ is overstated by $A_{0,E_k}/\beta$:
  $2.8/4 = 0.7$ at a buy, $0.2/4 = 0.05$ at a sell.
  The state gains its jump after the record, so the overstatement does not carry to the next row.

The residuals of the buys are differences of $\Lambda_0$ at consecutive buys.
Both ends of each difference are overstated by $0.7$, which cancels,
except in the first residual, whose lower end is $\Lambda_0(0) = 0$;
the rows at sells do not enter.
Only residual 0 moves, by $0.7$.

In [ ]:
2.8 / 4, 0.2 / 4

**The other choices.**
- (A) `[] []` on both lines: misses that C reads the post-jump count against the pre-jump state.
- (B) `[0] [0.7]` on both lines: misses that in B the two jumps cancel.
- (D) `[0, 1, 2] [0.7, 0.7, 0.7]`: misses that a constant overstatement cancels in a difference.
- (E) `[0, 1, 2] [0.7, 1.4, 2.1]`: takes the overstatement to accumulate;
  it does not, since `decayed` gains its jump after each record.

## 7.5 · `question.compensatorByRecursionSumOrQuadrature`

Three implementations of the compensator at the events,
timed on $n = 2000$, 4000, 8000 and 16000 events of the self-exciting pair.

**Answer: (D).**
At $n = 16000$ A is faster than B and than C by more than fivefold;
from $n = 8000$ to $n = 16000$ the times of A and C about double, and that of B grows more than threefold.

- **A** carries $(N, Z)$ from event to event at a fixed cost per event:
  the time of the path is linear in $n$, and doubles with $n$.
- **B** evaluates at each $T_k$ the closed form of the integral in the proof of Proposition 1.3.12 (`prop.compensatorClosedForm`),
  $$\int_0^{T_k} Z_{e'}(u)\,du = \sum_{j:\,T^{e'}_j < T_k}\frac{1 - e^{-\beta(T_k - T^{e'}_j)}}{\beta},$$
  over the $k$ earlier events.
  Event $k$ costs a fixed time plus one proportional to $k$, and the path $an + bn^2/2$.
  As $n$ grows the quadratic term takes over,
  and a doubling multiplies the time by more than 3, and towards 4.
  The sentence after eq. (1.3.38, `eq.update_of_exponential_intensity`) states the corresponding costs for the state $Z$,
  $O(n^2)$ over the path against $O(1)$ per event.
- **C** carries the state too, and integrates only over $(T_{k-1}, T_k)$, accumulating `total`:
  its cost per event is constant, and its time doubles with $n$.
  But each interval costs a call of `quad_vec`,
  which calls the Python function `intensity` at 21 points of each subinterval it makes:
  many steps of A.

**The other choices.**
- (A) C the fastest, `quad_vec` running compiled code:
  its integrand is a Python function, called at least 21 times per interval.
- (B) Each doubles: misses that B sums over the past.
- (C) C quadruples: takes C to integrate from 0 at every event;
  it integrates from the previous event, and accumulates.
- (E) B the fastest and A quadrupling:
  vectorising B's sum lowers its constant and not its growth, and A's step costs a constant.

## 7.6 · `question.groundResidualsCannotSeeTheTypes`

One path of the self-exciting pair, about 1900 buys,
tested against the self-exciting and the cross-exciting kernels,
$A = \begin{pmatrix} 0.2 & 2.8 \\ 2.8 & 0.2 \end{pmatrix}$ for the second.

**Answer: (B).** `True True`, then `True False`.

**The ground residuals.**
Summing eq. (1.3.15, `eq.compensatorClosedForm`) over $e$, with $\mathbf 1^\top A = (3, 3)$ under both kernels,
$$\Lambda_0(T) + \Lambda_1(T) = 0.5\,T + \tfrac34\big(N_{\mathfrak g}(T-) - \mathbf 1^\top Z(T)\big),$$
where $N_{\mathfrak g}$ counts the events of either type
and $\mathbf 1^\top Z(T) = \sum_{T_n < T}e^{-\beta(T - T_n)}$.
This is a function of the times alone:
the ground residuals are the same numbers under both kernels,
which see $A$ only through $\mathbf 1^\top A$, as the wait of Remark 1.3.45 (`remark.typeDraw`) does.
$\Lambda_0 + \Lambda_1$ is the compensator of $N_{\mathfrak g}$,
and Theorem 1.3.5 (`thm.meyer1971`) with one component makes its residuals $\mathrm{Exp}(1)$ under the true kernel:
`True` on both lines.

**The residuals of the buys.**
Under the self-exciting kernel, the true one, they are $\mathrm{Exp}(1)$ by Theorem 1.3.5 (`thm.meyer1971`): `True`.
Under the cross-exciting kernel, by eq. (1.3.15, `eq.compensatorClosedForm`),
$$\Lambda_0 = 0.25\,T + 0.05\,\big(N_0 - Z_0\big) + 0.7\,\big(N_1 - Z_1\big),$$
whose long-run increase per buy is still $0.25 + (0.2 + 2.8)/4 = 1$: the mean residual is about 1.
But this compensator rises after sells, not after buys,
so it misses the bursts of buys that follow buys:
the residuals inside a burst are too small, and those between bursts too large.
On about 1900 buys the test of the margin rejects: `False`.

In [ ]:
0.25 + (0.2 + 2.8) / 4

**The other choices.**
- (A) `True True` on both lines: misses that the compensator of the buys depends on the rows of $A$,
  not only on its column sums.
- (C) `False False` on the second line: takes the ground test to see the kernel beyond its column sums.
- (D) `False True` on the second line: exchanges what the two tests see.

## 7.7 · `question.antitheticWaitsPassTheMargin`

4000 residuals of a Poisson flow of rate 2, drawn independently (A)
and in antithetic pairs $U$, $1 - U$ (B).

**Answer: (E).** `True +0.0`, then `True -0.3`.

- **A**: independent $\mathrm{Exp}(1)$ residuals.
  The margin passes, and the lag-one autocorrelation is of order $1/\sqrt{4000} \approx 0.016$,
  printed `+0.0`.
- **B**: each residual is $-\log U$ or $-\log(1-U)$, an $\mathrm{Exp}(1)$ variable: the margin passes.
  Pairing $U$ with $1 - U$ balances the sample,
  so its empirical distribution function lies closer to the law than that of independent draws.

  The residuals come as $\tau_0, \tau_1, \tau_2, \dots$,
  with $(\tau_{2i}, \tau_{2i+1})$ antithetic and $(\tau_{2i+1}, \tau_{2i+2})$ independent.
  The adjacent pairs alternate between the two kinds,
  so the lag-one autocorrelation is about the average
  $$r \approx \tfrac12\Big(1 - \frac{\pi^2}{6}\Big) + \tfrac12 \cdot 0 \approx -0.32,$$
  printed `-0.3`.

Theorem 1.3.5 (`thm.meyer1971`) asserts the serial independence of the residuals,
which the comparison of the margin does not test, Remark 1.3.46 (`remark.marginsOnly`):
B passes the test of goodness of fit, and fails the theorem.

In [ ]:
1 - pi**2 / 6, (1 - pi**2 / 6) / 2, 1 / sqrt(4000)

**The other choices.**
- (A) `True +0.0` on both lines: misses the antithetic pairing.
- (B) `False -0.6`: takes every adjacent pair for antithetic, and the margin to see the dependence.
- (C) `True -0.6`: takes every adjacent pair for antithetic; only every other one is.
- (D) `False -0.3`: takes the test of the margin to see the dependence.

## 7.8 · `question.tradesReportedOnBothSides`

Under the model each side is a Poisson flow of rate 1, compensator $t$,
so the residuals are the gaps.
Three streams: two independent flows; the trade feed on both sides;
the feed with the sells moved by 1 ms.

**Answer: (C).** `True True 0.0`, then `True True 1.0` twice.

- **Line 1**, two independent flows: both margins pass,
  and the gaps of the two flows are independent: correlation about 0, `0.0`.
- **Line 2**, the trades as both buys and sells.
  Each side, on its own, is a Poisson flow of rate 1, and both margins pass.
  The two residual sequences are one sequence: correlation 1.
  The joint independence of Theorem 1.3.5 (`thm.meyer1971`) fails,
  and so does its hypothesis i, eq. (1.3.1, `eq.noCommonJumps`): the two components jump together.
- **Line 3**, the sells moved by 1 ms.
  The times are distinct, and hypothesis i holds.
  But `np.diff(trades + 0.001, prepend=0.0)` equals `np.diff(trades, prepend=0.0)`
  except in its first entry, larger by $0.001$:
  the sells are a function of the buys, and the correlation is 1.0 to one decimal.
  The margins pass.

The tests of the margins pass on all three streams, Remark 1.3.46 (`remark.marginsOnly`).

**The other choices.**
- (A) `0.0` on every line: misses that the two sides of the feed are one stream.
- (B) `0.0` on the third line: takes the millisecond to break the dependence; it breaks the ties only.
- (D) `False False` on the second line: takes the common jumps to spoil the margins;
  each margin, on its own, is a Poisson flow of rate 1.

## 7.9 · `question.fillsSharingATimestamp`

2000 market buys, a Poisson flow of rate 1, each recorded as two executions with the same time.
`horizon = orders[-1]`, about 2000 s,
so the rate of the executions is about $4000/2000 = 2$ and that of the distinct times about 1.

**Answer: (A).** 0.5, 0.25, and a number close to 0.

- **Line 1**, the executions at their own rate.
  `np.diff(fills, prepend=0.0)` alternates between a gap of the orders and 0:
  half the gaps are exactly zero, an atom of mass $1/2$,
  and the other half, rescaled by 2, are $2\,\mathrm{Exp}(1)$.
  For $x \geq 0$, $F_n(x) \approx 1 - \tfrac12 e^{-x/2}$, against $F(x) = 1 - e^{-x}$:
  $$F_n(x) - F(x) = e^{-x} - \tfrac12 e^{-x/2},$$
  which is $1/2$ at $x = 0$ and decreases to a minimum of $-1/16$, at $e^{-x/2} = 1/4$: $D = 1/2$.
  The count jumps by two at once, which eq. (1.3.1, `eq.noCommonJumps`) excludes.
- **Line 2**, the distinct times, which are the orders, at the rate of the executions:
  $2\,\mathrm{Exp}(1)$, $F_n(x) \approx 1 - e^{-x/2}$, and
  $$D = \max_x\big(e^{-x/2} - e^{-x}\big) = \tfrac14, \quad\text{at } x = 2\log 2.$$
- **Line 3**, the orders at their own rate: $\mathrm{Exp}(1)$,
  and $D$ of order $0.87/\sqrt{2000} \approx 0.02$ (G6).

In [ ]:
x = 2 * log(2)
0.25**2 - 0.25 / 2, exp(-x / 2) - exp(-x), 0.87 / sqrt(2000)

**The other choices.**
- (B) Close to 0 on every line: misses both the atom and the wrong rate.
- (C) 0.5, 0.5, close to 0: takes a rate wrong by a factor of 2 for as bad as the atom; it costs $1/4$.
- (D) 0.5, then close to 0 twice:
  misses that line 2 rescales the orders at the rate of the executions, twice theirs.
- (E) 0.25 on the first line: sees the factor 2 in the rate and misses the atom at zero.

## 7.10 · `question.twoSampleTestOnTheGapsOfOnePath`

300 comparisons by `stats.ks_2samp` of the gaps of one path of `ExponentialHawkes`
with those of one path of `OgataThinningHawkes`, about 500 events each,
on the self-exciting pair and on a Poisson flow of total rate 2.

**Answer: (B).** The first fraction is well above $0.01$, the second about $0.01$.

Both simulators draw the same law,
so a test that holds its level rejects about 1% of the comparisons.
Over 300 comparisons the standard error of the fraction at the level is
$\sqrt{0.01 \cdot 0.99/300} \approx 0.0057$.
- **The Poisson flow**, $A = 0$: the gaps of a path are independent $\mathrm{Exp}(2)$.
  `ks_2samp` sees two samples of independent draws from one law, and holds its level: about $0.01$.
- **The self-exciting pair**: the gaps along a path are dependent, the short ones clustered in bursts.
  By Theorem 1.3.5 (`thm.meyer1971`) it is the residuals, rescaled by the compensator, that are independent,
  not the gaps.
  The empirical distribution of the gaps of one path varies from path to path
  more than that of 500 independent gaps would,
  while `ks_2samp` measures the distance between two paths against the spread of independent samples:
  it overstates its evidence, and rejects well above 1%.

One statistic per independent path restores the level.

In [ ]:
sqrt(0.01 * 0.99 / 300), 2 * 250

**The other choices.**
- (A) Both about $0.01$: takes `ks_2samp` to hold its level on dependent samples.
- (C) Both well above: takes the two simulators to draw different laws; on the Poisson flow they agree.
- (D) The first about $0.01$, the second well above: exchanges the dependent and the independent case.